# 02 · Tool Calling 的本質

「模型會呼叫工具」聽起來很神奇。拆穿了，它只是四步，而且**每一步都是我們寫的**：

1. **描述工具**：system prompt 告訴模型有哪些工具、用什麼格式呼叫。
2. **模型吐結構化文字**：它判斷要用工具時，就照約定吐出來——它沒真的執行任何東西。
3. **我們解析 + 執行**：用 Python 解析那段文字，真的去呼叫對應函式。
4. **把結果餵回去**：把工具回傳值塞回對話，請模型給最終答案。

真正碰到時鐘、碰到世界的，是步驟 3 裡**你寫的 Python**。

## 1. 載入模型

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

## 2. 定義第一個工具：一支真正的時鐘

In [ ]:
from datetime import datetime, timezone, timedelta


def get_current_time() -> str:
    """回傳台北現在時間（模型沒有時鐘，這是它拿不到的真實世界資訊）。"""
    tz = timezone(timedelta(hours=8))
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")


def calculator(expression: str) -> str:
    """安全地算一個算式。只允許數字與 + - * / ( ) . %。"""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression or ""):
        return "錯誤：只接受數字與 + - * / ( ) . %"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"錯誤：{e}"

In [ ]:
print("現在真實時間：", get_current_time())

## 3. 教模型用約定格式呼叫工具

我們約定：需要工具時，輸出 `Action: 工具名` 與 `Action Input: {JSON 參數}`。

In [ ]:
SYSTEM = """你是一個會使用工具的助手。你有以下工具：

- get_current_time()：回傳現在的台北時間，不需要參數。

當你需要工具時，嚴格用這個格式輸出（不要說別的）：
Action: 工具名稱
Action Input: JSON 參數（沒有參數就寫 {})

不需要工具時，直接回答即可。"""

In [ ]:
def parse_action(text):
    """從模型輸出抓出 (工具名, 參數dict)；抓不到回 (None, None)。"""
    m_name = re.search(r"Action:\s*([A-Za-z_]\w*)", text)
    if not m_name:
        return None, None
    args = {}
    m_args = re.search(r"Action Input:\s*(\{.*\})", text, re.DOTALL)
    if m_args:
        try:
            args = json.loads(m_args.group(1))
        except Exception:
            args = {}
    return m_name.group(1), args

## 4. 第一輪：讓模型決定要不要用工具

In [ ]:
question = "現在台北時間幾點？"
messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": question}]
reply = chat(messages)
print("模型第一次輸出：\n" + reply)

## 5. 解析 → 執行 → 餵回 → 最終答案

把四步串完。

In [ ]:
TOOLS = {"get_current_time": get_current_time}

name, args = parse_action(reply)
if name in TOOLS:
    observation = TOOLS[name](**args)            # 步驟 3：真的執行
    print(f"執行 {name}() → {observation}")
    messages.append({"role": "assistant", "content": reply})
    messages.append({"role": "user",
                     "content": f"Observation: {observation}\n請根據這個結果回答原問題。"})
    final = chat(messages)                        # 步驟 4：餵回拿最終答案
    print("\n最終答案：\n" + final)
else:
    print("模型沒呼叫工具，直接回答：\n" + reply)

## 小結

- Tool calling = **結構化文字 + 你寫的解析器**，沒有魔法。
- 我們手刻完了「描述 → 吐字 → 解析執行 → 餵回」一輪。
- 但這只跑了**一輪、一個工具**。真實任務要連續好幾步。

下一課：把這一輪一般化成 **ReAct 迴圈**。